# DT401 Week 5 Section 2a: Supervised Model — Linear Regression

**Module:** Data Fundamentals
**Week:** 5 of 7
**Section:** 2a of 3
**Estimated Time:** 30 minutes

---

## Learning Objective

Build and evaluate a supervised linear regression model using real weather data.

By the end of this notebook, you will be able to:
- Explain what a supervised regression model is and when to use it
- Prepare data and build a linear regression model in Python
- Interpret model coefficients in plain business language
- Calculate and interpret R-squared and RMSE
- Distinguish between a well-fitted and a poorly-fitted model

---

## Context Recap

In Section 1, you learned the six-stage data analysis lifecycle. This notebook focuses on **Stage 4: Model Building** — specifically supervised regression. You will build a model using three years of real London weather data to predict the apparent (feels-like) temperature from actual temperature and wind speed.

**Weather data from Open-Meteo API (https://open-meteo.com) — London, UK (51.49°N, 0.16°W), Jan 2023–Jan 2026**

---

## How to Use This Notebook

This notebook contains a mix of instructional text, Python code cells, and interactive exercises. Follow the instructions in each section, run the code cells to see outputs, and complete the exercises to reinforce your learning.

You will find several types of boxes throughout the notebook to guide your learning:

<div class="alert alert-block alert-warning">
<b>Reminder boxes:</b> <br>
These boxes contain important reminders about how this section relates to the overall module and your learning journey.
</div>

<div class="alert alert-block alert-info">
<b>Reflection boxes:</b> <br>
These have pointers as to what we should be looking for or comments on how to approach the topic discussed.
</div>

<div class="alert alert-block alert-success">
<b>Task boxes:</b> <br>
These have questions or tasks you should complete before progressing.

These may include hidden hints or solutions.
<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Hints help guide your thinking without giving away the full answer.

</details>

</div>

---

## Part 1: What Is a Supervised Model?

### Visual Overview

```mermaid
graph LR
    A[Temperature °C<br/>Independent Variable] --> C[Linear Regression Model]
    B[Wind Speed km/h<br/>Independent Variable] --> C
    C --> D[Apparent Temperature °C<br/>Dependent Variable]

    style A fill:#e1f5ff
    style B fill:#e1f5ff
    style C fill:#fff4e1
    style D fill:#e1ffe1
```

**What this diagram shows:**
A supervised regression model takes one or more independent variables (inputs) and predicts a single continuous dependent variable (output). The model learns the relationship from historical data where both inputs and outputs are known.

**Why this matters in your role:**
Regression answers "how much?" questions — apparent temperature, patient waiting time, sales volume, delivery cost. Any continuous outcome driven by measurable inputs is a regression candidate.

---

### Business Scenario

An outdoor events company plans concerts, festivals, and sporting events across London. Attendee comfort, staffing decisions, and contingency planning all depend not on the measured air temperature but on the **apparent temperature** — the 'feels-like' value that accounts for wind chill.

Both temperature and wind speed are available from weather forecasts in advance of any event. We can therefore build a regression model that predicts apparent temperature from these two inputs, giving the company a reliable comfort forecast for any planned event day.

---

### Approach

We use the Open-Meteo historical dataset for London (January 2023 to January 2026, 26,328 hourly observations). Both measured temperature and wind speed are available from weather forecasts before an event takes place, making them suitable independent variables.

---

### What Is Linear Regression?

Linear regression models a straight-line relationship between one or more independent variables and a continuous dependent variable. The algorithm calculates the **line of best fit** — the line that minimises the total squared error between predictions and actual values across all observations.

The scatterplot below shows the relationship between temperature and apparent temperature across the full London dataset. A strong positive trend is clearly visible:

![Temperature vs Apparent Temperature with regression line](../images/reg_scatter.png)

The model produces a **fitted equation** from this data:

> **Apparent temperature = −2.29 + 1.167 × temperature − 0.152 × wind speed**

**Interpreting the equation:**
- For every 1°C rise in temperature, apparent temperature rises by **1.17°C**
- For every 10 km/h increase in wind speed, apparent temperature falls by **1.52°C** — the wind chill effect
- At 0°C with no wind, the baseline apparent temperature is **−2.29°C**

---

### Requirements

For linear regression you need:

- A **dependent variable** — the one you predict. Must be **continuous** (a number). Here: apparent temperature (°C).
- One or more **independent variables** — the predictors. Must be **numerical**. Here: actual temperature (°C) and wind speed (km/h).
- The dependent variable should have a **linear relationship** with each independent variable — confirmed by the scatterplot above.

| Variable | Column | Role |
| :--- | :--- | :--- |
| Apparent temperature (°C) | `apparent_temperature` | Dependent variable — what we predict |
| Actual temperature (°C) | `temperature_2m` | Independent variable |
| Wind speed (km/h) | `wind_speed_10m` | Independent variable |

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Load weather dataset
weather_data = pd.read_csv('../open-meteo-51.49N0.16W23m.csv', skiprows=3, low_memory=False)
weather_data.columns = [c.split(' ')[0].strip() for c in weather_data.columns]
weather_data = weather_data.rename(columns={'time': 'date'})

# Keep only hourly rows (YYYY-MM-DDTHH:MM format)
# The CSV contains a second daily-data section appended after the hourly data
weather_data = weather_data[weather_data['date'].str.match(r'^\d{4}-\d{2}-\d{2}T', na=False)].reset_index(drop=True)

# Convert all measurement columns to numeric
for col in weather_data.columns[1:]:
    weather_data[col] = pd.to_numeric(weather_data[col], errors='coerce')
weather_data['wind_speed_10m'] = weather_data['wind_speed_10m'] * 1.60934

print(f"Dataset loaded: {len(weather_data):,} hourly observations")
print(f"Date range: {weather_data['date'].min()} to {weather_data['date'].max()}")
weather_data[['date', 'temperature_2m', 'wind_speed_10m', 'apparent_temperature']].head()

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
A model is always a simplification. The apparent temperature in this dataset is itself a meteorological calculation — our model is learning to approximate it from its two main inputs. The goal is not a perfect model, but a model useful enough to support real decisions.
</div>

---

## Part 2: Building the Regression Model

### Step 1 — Prepare the Data

In [ ]:
# Select the three relevant columns and drop any rows with missing values
regression_data = weather_data[['temperature_2m', 'wind_speed_10m', 'apparent_temperature']].copy()
regression_data = regression_data.dropna()

print(f"Clean dataset: {len(regression_data):,} observations")
print()
print("Summary statistics:")
regression_data.describe().round(2)

<div class="alert alert-block alert-info">
<b>What to look for:</b><br>

- <b>temperature_2m</b>: ranges from about −5°C to 34°C — London's full seasonal range
- <b>wind_speed_10m</b>: mean around 13–14 km/h, peaks above 50 km/h in storms
- <b>apparent_temperature</b>: will typically be <i>lower</i> than actual temperature due to wind chill

Notice that apparent temperature has a lower mean than actual temperature — wind chill is pulling it down on average.
</div>

### Step 2 — Define Variables and Fit the Model

In [ ]:
# Define independent variables (inputs) and dependent variable (output)
X = regression_data[['temperature_2m', 'wind_speed_10m']]  # Independent variables
y = regression_data['apparent_temperature']                  # Dependent variable

# Fit the linear regression model
model = LinearRegression()
model.fit(X, y)

# Display model parameters
print("=== MODEL PARAMETERS ===")
print(f"Temperature coefficient : {model.coef_[0]:.4f}")
print(f"Wind speed coefficient  : {model.coef_[1]:.4f}")
print(f"Intercept (constant)    : {model.intercept_:.4f}")
print()
print("=== MODEL EQUATION ===")
print(f"Apparent temp = {model.intercept_:.2f} + {model.coef_[0]:.3f} x temperature + {model.coef_[1]:.3f} x wind_speed")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We defined <b>temperature</b> and <b>wind speed</b> as independent variables — the inputs
- We defined <b>apparent temperature</b> as the dependent variable — what we are predicting
- The model calculated two coefficients and one intercept to best fit the data

<b>Interpreting the coefficients:</b>
<ul>
<li>Temperature coefficient ≈ +1.17: for every 1°C rise in temperature, apparent temperature rises by 1.17°C</li>
<li>Wind speed coefficient ≈ −0.15: for every 10 km/h increase in wind, apparent temperature falls by ~1.5°C</li>
<li>Intercept ≈ −2.29: the predicted apparent temperature when both inputs are zero</li>
</ul>

<b>Key lesson:</b> The sign of a coefficient tells you the direction of the relationship. Negative wind speed coefficient confirms the wind chill effect — wind reduces perceived temperature.
</div>

### Step 3 — Visualise the Model

In [ ]:
# Generate predictions for all observations
predictions = model.predict(X)

# Sample 500 points for plotting
np.random.seed(42)
idx = np.random.choice(len(y), 500, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Temperature vs Apparent Temperature with regression line
avg_wind = regression_data['wind_speed_10m'].mean()
x_line = np.linspace(regression_data['temperature_2m'].min(),
                     regression_data['temperature_2m'].max(), 100)
y_line = model.intercept_ + model.coef_[0] * x_line + model.coef_[1] * avg_wind

axes[0].scatter(regression_data['temperature_2m'].iloc[idx], y.iloc[idx],
                alpha=0.4, s=10, color='steelblue', label='Observed')
axes[0].plot(x_line, y_line, 'r-', lw=2, label=f'Regression line (avg wind = {avg_wind:.0f} km/h)')
axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Apparent Temperature (°C)')
axes[0].set_title('Temperature vs Apparent Temperature')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Right panel: Actual vs Predicted
axes[1].scatter(y.iloc[idx], predictions[idx], alpha=0.4, s=10, color='steelblue')
axes[1].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Perfect prediction (y = x)')
axes[1].set_xlabel('Actual Apparent Temperature (°C)')
axes[1].set_ylabel('Predicted Apparent Temperature (°C)')
axes[1].set_title('Actual vs Predicted')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Left: strong positive trend confirms temperature drives apparent temperature")
print("Right: points close to the red line = accurate predictions")

---

### Wind Chill Visualised

The wind chill relationship is confirmed by plotting wind speed against the gap between measured temperature and apparent temperature:

![Wind speed vs wind chill gap](../images/reg_wind_chill.png)

As wind speed increases, the gap between actual and perceived temperature grows systematically — confirming that wind speed contributes genuine predictive value beyond temperature alone. This is the physical basis for including wind speed as an independent variable in the model.

### Step 4 — Measure Model Performance

To measure how well the model performs, we compare its predictions against the actual recorded values. For each observation we have three values:

- The actual temperature and wind speed (the independent variable inputs)
- The actual apparent temperature recorded at that time (the dependent variable)
- The apparent temperature the model predicts from those two inputs

**Sample observations — comparing actual vs predicted:**

| Observation | Temperature (°C) | Wind speed (km/h) | Actual apparent temp (°C) | Predicted (°C) | Error |
|:-----------:|:----------------:|:-----------------:|:-------------------------:|:--------------:|:-----:|
| 1 | 14.9 | 11.3 | 14.0 | 13.4 | 0.6 |
| 2 | 7.6 | 20.0 | 4.0 | 3.5 | 0.5 |
| 3 | 12.6 | 9.3 | 9.9 | 11.0 | −1.1 |
| 4 | 16.3 | 10.1 | 15.1 | 15.2 | −0.1 |
| 5 | 14.4 | 24.9 | 10.6 | 10.7 | −0.1 |
| **Total** | | | **53.6** | **53.8** | **−0.2** |

The individual errors include both positive and negative values — summing them directly produces a misleadingly small total. To solve this, we **square each error** (making all values positive) then sum them to produce the **Sum Squared Error (SSE)**:

| Observation | Temperature (°C) | Wind speed (km/h) | Actual apparent temp (°C) | Predicted (°C) | Squared error |
|:-----------:|:----------------:|:-----------------:|:-------------------------:|:--------------:|:------------:|
| 1 | 14.9 | 11.3 | 14.0 | 13.4 | 0.36 |
| 2 | 7.6 | 20.0 | 4.0 | 3.5 | 0.25 |
| 3 | 12.6 | 9.3 | 9.9 | 11.0 | 1.21 |
| 4 | 16.3 | 10.1 | 15.1 | 15.2 | 0.01 |
| 5 | 14.4 | 24.9 | 10.6 | 10.7 | 0.01 |
| **Total** | | | **53.6** | **53.8** | **1.84** |

From SSE we derive two further metrics:

1. **Mean Squared Error (MSE):** Divide SSE by the number of observations → 1.84 ÷ 5 = **0.37**
2. **Root Mean Squared Error (RMSE):** Take the square root to return to the original units → √0.37 ≈ **0.61°C**

RMSE is expressed in the same units as the dependent variable (°C), making it directly interpretable as a typical prediction error. Across all 26,328 observations in the full dataset, the model produces an **RMSE of 0.68°C**.

When a computer builds a model, it tests many variants and optimises the parameters until it finds the **lowest SSE** — this is exactly what `LinearRegression().fit()` does in a single step. Run the code below to calculate R-squared and RMSE for the full dataset:

In [ ]:
# Calculate R-squared and RMSE
r2 = r2_score(y, predictions)
rmse = np.sqrt(mean_squared_error(y, predictions))

print("=== MODEL PERFORMANCE ===")
print(f"R-squared : {r2:.3f}  ({r2*100:.1f}% of variance explained)")
print(f"RMSE      : {rmse:.2f} deg C")
print()
print("Interpretation:")
print(f"- The model explains {r2*100:.1f}% of the variation in apparent temperature")
print(f"- Typical prediction error: +/- {rmse:.2f} deg C")
print()
print("For context: an RMSE below 1 deg C means predictions are reliable")
print("for comfort planning — most people cannot perceive differences smaller than 1 deg C.")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- <b>R-squared</b> measures how much of the variation in apparent temperature the model captures. A value of 0.991 (99.1%) means the model is an excellent fit.
- <b>RMSE</b> measures the typical prediction error in the original units (°C). A value below 1°C means the model is accurate enough to use in practice.

<b>Key lesson:</b> High R² + low RMSE = strong model. This result is achievable because temperature and wind speed have a genuine physical relationship with apparent temperature. Models built on coincidental correlations produce low R² and high RMSE.
</div>

### R-squared in Detail

**R-squared** (R²) measures how much of the variation in the dependent variable the model explains. It ranges from 0 to 1:

- **R² = 1**: the model explains all variation — a perfect fit
- **R² = 0**: the model explains none of the variation — no better than always predicting the mean value

For the apparent temperature model, **R² = 0.991** — temperature and wind speed together explain 99.1% of the variation in apparent temperature. This reflects a genuine physical relationship: apparent temperature is directly driven by these two variables through the well-understood wind chill effect.

The actual vs predicted scatter confirms the accuracy visually — points fall very close to the perfect prediction line (y = x):

![Actual vs predicted apparent temperature](../images/reg_actual_vs_predicted.png)

---

### Interpreting Error in Context

**R-squared and RMSE answer different questions:**
- R² tells you how much of the overall pattern the model has captured
- RMSE tells you how large the typical error is, expressed in the original business units (°C)
- Both are reported in practice — both are calculated directly using `r2_score` and `mean_squared_error` from `sklearn.metrics`

**What these numbers mean operationally:**
An R² of 99.1% and RMSE of 0.68°C means this model is highly useful for operational decision-making. A prediction error of less than 1°C is small enough to make reliable attendee comfort and staffing decisions. This result is achievable because temperature and wind speed have a strong, direct causal relationship with apparent temperature — models built on genuine causal relationships produce high R² and low RMSE; those built on coincidental correlations do not.

---

<div class="alert alert-block alert-warning">
<b>Bridge to Section 2b:</b> <br>
The SSE principle — squaring errors and summing them to produce a single measure of model fit — also appears in k-means clustering. There, it is called <b>inertia</b>, and is plotted against the number of clusters k to identify the point of diminishing returns. This is the <b>elbow method</b>, which you will apply in Section 2b.
</div>

---

## Part 2: Modified Task — Single-Variable Model

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Build a simpler regression model that predicts apparent temperature using <b>temperature alone</b> (no wind speed). Compare its R-squared and RMSE to the two-variable model above.

Which performs better? What does this tell you about the value of including wind speed?

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Steps:
1. Define X_simple = regression_data[['temperature_2m']]  (note: double brackets keep it as DataFrame)
2. y remains the same (apparent_temperature)
3. Fit a new LinearRegression() model
4. Calculate r2_score and mean_squared_error on the predictions
5. Print and compare to the two-variable results above

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
X_simple = regression_data[['temperature_2m']]

simple_model = LinearRegression()
simple_model.fit(X_simple, y)
simple_preds = simple_model.predict(X_simple)

simple_r2 = r2_score(y, simple_preds)
simple_rmse = np.sqrt(mean_squared_error(y, simple_preds))

print("Single-variable model (temperature only):")
print(f"  R-squared : {simple_r2:.3f} ({simple_r2*100:.1f}%)")
print(f"  RMSE      : {simple_rmse:.2f} deg C")
print()
print("Two-variable model (temperature + wind speed):")
print(f"  R-squared : {r2:.3f} ({r2*100:.1f}%)")
print(f"  RMSE      : {rmse:.2f} deg C")
print()
print("Adding wind speed improves RMSE from ~1.3 deg C to ~0.7 deg C")
print("Wind speed captures the wind chill effect the single-variable model misses.")
```

</details>

</div>

In [ ]:
X_simple = regression_data[['temperature_2m']]

simple_model = LinearRegression()
simple_model.fit(X_simple, y)
simple_preds = simple_model.predict(X_simple)

simple_r2 = r2_score(y, simple_preds)
simple_rmse = np.sqrt(mean_squared_error(y, simple_preds))

print("Single-variable model (temperature only):")
print(f"  R-squared : {simple_r2:.3f} ({simple_r2*100:.1f}%)")
print(f"  RMSE      : {simple_rmse:.2f} deg C")
print()
print("Two-variable model (temperature + wind speed):")
print(f"  R-squared : {r2:.3f} ({r2*100:.1f}%)")
print(f"  RMSE      : {rmse:.2f} deg C")
print()
print("Adding wind speed improves RMSE from ~1.3 deg C to ~0.7 deg C")
print("Wind speed captures the wind chill effect the single-variable model misses.")


print("Single-variable model results:")

**Expected output:** R-squared and RMSE for the single-variable model

**Success check:** The two-variable model should have a higher R² and lower RMSE. You can explain why adding wind speed improves the model.

---

## Part 3: Independent Task — Model Selection for Your Workplace

### Your Challenge

You work for a local council managing community services. The council has data on:
- Service requests (type, location, priority, resolution time)
- Resource allocation (staff hours, equipment, costs)
- Citizen satisfaction scores
- Geographic data (neighbourhood demographics, service accessibility)

**Your manager asks: "Can data analysis help us improve service delivery?"**

Design **three** different analytical approaches:

1. **Regression model:** What continuous variable could you predict? What predictors would you use? What business decision does this support?
2. **Classification model:** What categories could you predict? What features would indicate membership? What actions follow from each prediction?
3. **Clustering model:** What natural groupings might exist? What variables define similarity? How would the clusters be used operationally?

**For each model, define:**
- The business problem it addresses
- The dependent variable (regression/classification) or clustering variables
- The independent variables / features
- The actionable insights the model provides
- How you would measure success

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

**Regression:** Predict service resolution time (hours) from request type, priority, neighbourhood, time of day.
**Classification:** Predict priority level (Low / Medium / High / Urgent) from request description, location, historical patterns.
**Clustering:** Group neighbourhoods by service volume, response time, satisfaction scores.

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
model_designs = {
    'regression': {
        'business_problem': 'Predict service resolution time to set accurate citizen expectations',
        'dependent_variable': 'Resolution time (hours)',
        'independent_variables': ['Request type', 'Priority', 'Neighbourhood', 'Time of day', 'Staff on duty'],
        'actionable_insights': 'Show citizens realistic timeframes at submission; flag high-delay scenarios for resource reallocation',
        'success_metric': 'RMSE < 4 hours'
    },
    'classification': {
        'business_problem': 'Auto-prioritise service requests so urgent issues are handled first',
        'categories': ['Low', 'Medium', 'High', 'Urgent'],
        'features': ['Request description (text)', 'Location risk', 'Historical patterns', 'Time since last service'],
        'actionable_insights': 'Route Urgent requests to senior staff automatically; batch Low requests for scheduled maintenance',
        'success_metric': '85% classification accuracy; all Urgent predictions reviewed by senior staff'
    },
    'clustering': {
        'business_problem': 'Identify neighbourhoods with similar service needs for targeted resource allocation',
        'clustering_variables': ['Service volume', 'Average response time', 'Satisfaction scores', 'Demographics'],
        'actionable_insights': 'High-need clusters get proactive outreach and additional resources; low-need clusters optimise for efficiency',
        'success_metric': 'Clusters show distinct profiles; satisfaction improves 20% in high-need clusters after reallocation'
    }
}
```

</details>

In [ ]:
# Your model designs — complete this for your portfolio
model_designs = {
    'regression': {
        'business_problem': '',
        'dependent_variable': '',
        'independent_variables': [],
        'actionable_insights': '',
        'success_metric': ''
    },
    'classification': {
        'business_problem': '',
        'categories': [],
        'features': [],
        'actionable_insights': '',
        'success_metric': ''
    },
    'clustering': {
        'business_problem': '',
        'clustering_variables': [],
        'actionable_insights': '',
        'success_metric': ''
    }
}

for model_type, design in model_designs.items():
    print(f"\n=== {model_type.upper()} MODEL ===")
    for key, value in design.items():
        print(f"  {key}: {value}")

**Success criteria:**
- All three model types designed with clear business problems
- Appropriate variables/features identified for each approach
- Actionable insights clearly stated — not just technical outputs
- Success metrics are specific and measurable

**Portfolio note:** This model selection task maps to MLO 3. Include it in your Module Challenge Part A portfolio.

---

## Reflection

Reflect on the following question and record your thoughts in the cell below. Ground your answer in an example from your workplace, using **approximately 150 words** in total.

Your answer here forms part of your Module Challenge Part A portfolio and is assessed against **MLO3:** *Select and apply appropriate data-gathering methods for a given data analysis scenario*.

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your response. You can develop your answer further before your final submission.

<div class="alert alert-block alert-info">

1. Define, in your own words, the difference between a data model and a measurement. Using this notebook's regression model as an example, explain how you would apply these concepts in your workplace to understand the profitability of a product, service or project.

</div>

*Write your responses here.*

1. 

References (if applicable):

---

## Summary

**You have completed Week 5 Section 2a.** You can now:

- Explained what a supervised regression model is, distinguishing the independent variables (temperature, wind speed) from the continuous dependent variable (apparent temperature)
- Built a two-variable linear regression model with `sklearn`'s `LinearRegression`, interpreting its coefficients (temperature +1.17, wind speed -0.15) and evaluating it with `r2_score` and `mean_squared_error` (R-squared 0.991, RMSE 0.68 deg C)
- Built a single-variable model using temperature alone and compared its R-squared and RMSE against the two-variable model, showing the accuracy gained by adding wind speed
- Designed three analytical approaches (regression, classification, clustering) for a local council service-delivery problem, defining variables, features, and success metrics for each

**Key takeaways:**

- RMSE is derived from Sum Squared Error: divide by the number of observations to get Mean Squared Error, then take the square root to return the error to the dependent variable's original units
- R-squared and RMSE answer different questions: R-squared shows how much of the variation the model explains, RMSE shows the typical size of its error. Report both together, not one alone

**What comes next:**

- **Section 2b: Unsupervised Model, K-Means Clustering** carries forward the Sum Squared Error principle used here: in k-means it is called inertia, and the elbow method uses it to choose the number of clusters.

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 5 Section 2a Linear Regression"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.5 component on Aptem
5. Open `DT401_Week5_Section2b.ipynb` when ready to continue